In [68]:
from dotenv import find_dotenv, load_dotenv
from langchain.output_parsers import ResponseSchema, StructuredOutputParser
from langchain.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

In [2]:
_ = load_dotenv(find_dotenv("../creds/.env"), verbose=True)

In [3]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

In [4]:
template_string = (
    "Translate the text that is delimited by triple backticks into a style that is {style}. text: ```{text}```"
)

In [5]:
customer_style = "American English in a calm and respectful tone"
customer_text = """
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse, \
the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!
"""

In [6]:
support_style = "a polite tone that speaks in English Pirate"
support_text = """Hey there customer, the warranty does not cover cleaning expenses for your kitchen because it's your fault that \
you misused your blender by forgetting to put the lid on before starting the blender. Tough luck! See ya!
"""

In [7]:
template = ChatPromptTemplate.from_template(template_string)

In [9]:
customer_messages = template.format_messages(style=customer_style, text=customer_text)
customer_response = llm.invoke(customer_messages)
customer_response

AIMessage(content="Okay, I understand you're quite upset that your blender lid came off and made a mess of your kitchen with smoothie. That sounds really frustrating! And it's definitely disappointing to hear that the warranty won't cover the cleaning costs. I'll do my best to help you figure out what options you have.", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--bddd6956-d3c6-476d-b92c-a057cc00d618-0', usage_metadata={'input_tokens': 82, 'output_tokens': 66, 'total_tokens': 148, 'input_token_details': {'cache_read': 0}})

In [10]:
support_messages = template.format_messages(style=support_style, text=support_text)
support_response = llm.invoke(support_messages)
support_response

AIMessage(content="Ahoy there, matey!\n\n'Tis with a heavy heart I must inform ye that yer warranty, alas, doesn't cover the shinin' up o' yer galley. Seems yer blender took a bit o' a spill, and 'twas due to a wee oversight on yer part – forgettin' the lid, ye say? Aye, a common mistake, but one that befalls ye, not the warranty.\n\nSo, while I be feelin' for yer predicament, the responsibility for the mess rests squarely on yer shoulders. Hard luck, I say!\n\nFair winds and followin' seas, but I be off now! Arrr!", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--fbc750db-934b-43f6-94ad-98d997d6a3e3-0', usage_metadata={'input_tokens': 72, 'output_tokens': 136, 'total_tokens': 208, 'input_token_details': {'cache_read': 0}})

### Output parsers

In [ ]:
customer_review = """
This leaf blower is pretty amazing.  It has four settings: candle blower, gentle breeze, windy city, and tornado.
It arrived in two days, just in time for my wife's anniversary present.
I think my wife liked it so much she was speechless. So far I've been the only one using it, and I've been
using it every other morning to clear the leaves on our lawn. It's slightly more expensive than the other leaf blowers
out there, but I think it's worth it for the extra features.
"""

review_template = """\
For the following text, extract the following information:\

gift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.\

delivery_days: How many days did it take for the product to arrive? If this information is not found, output -1.\

price_value: Extract any sentences about the value or price, and output them as a comma separated Python list.\

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}
"""

In [55]:
review_prompt_template = ChatPromptTemplate.from_template(review_template)
messages = review_prompt_template.format_messages(text=customer_review)
llm.invoke(messages).content

'```json\n{\n  "gift": true,\n  "delivery_days": 2,\n  "price_value": [\n    "It\'s slightly more expensive than the other leaf blowers\\nout there, but I think it\'s worth it for the extra features."\n  ]\n}\n```'

### Parse the LLM output string into python dictionary

In [77]:
gift_schema = ResponseSchema(
    name="gift",
    description="Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.",
    type="bool",
)
delivery_days_schema = ResponseSchema(
    name="delivery_days",
    description="How many days did it take for the product to arrive? If this information is not found - output -1.",
    type="integer",
)
price_value_schema = ResponseSchema(
    name="price_value",
    description="Extract any sentences about the value or price, and output them as comma-separated Python list.",
    type="list[string]",
)

response_schemas = [gift_schema, delivery_days_schema, price_value_schema]

In [79]:
response_schemas

[ResponseSchema(name='gift', description='Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.', type='bool'),
 ResponseSchema(name='delivery_days', description='How many days did it take for the product to arrive? If this information is not found - output -1.', type='integer'),
 ResponseSchema(name='price_value', description='Extract any sentences about the value or price, and output them as comma-separated Python list.', type='list[string]')]

In [89]:
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
output_parser

StructuredOutputParser(response_schemas=[ResponseSchema(name='gift', description='Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.', type='bool'), ResponseSchema(name='delivery_days', description='How many days did it take for the product to arrive? If this information is not found - output -1.', type='integer'), ResponseSchema(name='price_value', description='Extract any sentences about the value or price, and output them as comma-separated Python list.', type='list[string]')])

In [92]:
format_instructions = output_parser.get_format_instructions()
format_instructions

'The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":\n\n```json\n{\n\t"gift": bool  // Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.\n\t"delivery_days": integer  // How many days did it take for the product to arrive? If this information is not found - output -1.\n\t"price_value": list[string]  // Extract any sentences about the value or price, and output them as comma-separated Python list.\n}\n```'

In [93]:
review_template = """
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price, and output them as a comma separated Python list.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}

{format_instructions}
"""

prompt = ChatPromptTemplate.from_template(template=review_template)
messages = prompt.format_messages(text=customer_review, format_instructions=format_instructions)
output_parser.parse(llm.invoke(messages).content)

{'gift': True,
 'delivery_days': 2,
 'price_value': ["It's slightly more expensive than the other leaf blowers\nout there, but I think it's worth it for the extra features."]}